# Train Protein-Aware E(n) Equivariant GNN on LP-PDBBind

This notebook trains the custom `GNNPredictor` with **ESM-2 protein cross-attention** on LP-PDBBind.

**Key difference from v1**: `use_protein_encoder=True` activates the `ProteinEncoder` module
which cross-attends ligand EGNN features with per-residue ESM-2 embeddings (`facebook/esm2_t33_650M_UR50D`, 1280-dim).

**Hardware:** Requires A100 GPU (ESM-2 650M + GNN training).

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install -q torch-geometric rdkit transformers accelerate

## 2. Mount Google Drive, Clone Repo & Download Data

In [ ]:
from google.colab import drive
import os, urllib.request

drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/agentic-vlm'

if not os.path.exists(PROJECT_DIR):
    !git clone https://github.com/handeboyaci/agentic-vlm.git "$PROJECT_DIR"
else:
    print('Repo already cloned, pulling latest...')
    !cd "$PROJECT_DIR" && git pull

%cd "$PROJECT_DIR"

os.makedirs('data/pdbbind_deepchem/raw', exist_ok=True)
csv_path = 'data/pdbbind_deepchem/raw/LP_PDBBind.csv'
if not os.path.exists(csv_path):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/THGLab/LP-PDBBind/master/dataset/LP_PDBBind.csv',
        csv_path
    )
    print('Downloaded LP_PDBBind.csv')
else:
    print('LP_PDBBind.csv already exists.')

## 3. Dataset Preprocessing (with ESM-2 Embeddings)

Setting `precompute_esm=True` runs ESM-2 on each protein sequence during
data processing and stores the embeddings in `data.protein_emb`.

**First run takes ~30-60 min** to compute ESM-2 for all ~19K sequences.
After that, embeddings are cached to disk at `data/esm_cache/`.

In [ ]:
import torch
from torch_geometric.loader import DataLoader
from data.lp_pdbbind import LPPDBBind

# Delete processed/ dir to force re-processing with ESM-2:
#   !rm -rf data/pdbbind_deepchem/processed/

train_full = LPPDBBind(
    root='data/pdbbind_deepchem',
    split='train',
    precompute_esm=True,  # <-- NEW: compute ESM-2 embeddings
)
test_dataset = LPPDBBind(
    root='data/pdbbind_deepchem',
    split='test',
    precompute_esm=True,
)

train_full = train_full.shuffle()
val_size = int(0.1 * len(train_full))
val_dataset = train_full[:val_size]
train_dataset = train_full[val_size:]

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Check if ESM-2 embeddings are present
sample = train_dataset[0]
has_esm = hasattr(sample, 'protein_emb') and sample.protein_emb is not None
print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}')
print(f'ESM-2 embeddings present: {has_esm}')
if has_esm:
    print(f'ESM-2 embedding shape: {sample.protein_emb.shape}')

## 4. Initialize the Protein-Aware EGNN Model

In [ ]:
from models.gnn_predictor import GNNPredictor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on {device}")

model = GNNPredictor(
    atom_feat_dim=43,
    hidden_dim=128,
    n_layers=4,
    dropout=0.2,
    use_protein_encoder=True,  # <-- NEW: activate ProteinEncoder
).to(device)

# Count params
total = sum(p.numel() for p in model.parameters())
prot = sum(p.numel() for n, p in model.named_parameters() if 'protein_encoder' in n)
print(f'Total parameters: {total:,} (protein encoder: {prot:,})')

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)
criterion = torch.nn.MSELoss()

## 5. Training Loop (Protein-Aware)

The key change: we pass `protein_embs` to the model forward.
If a sample has `protein_emb`, we use it; otherwise `None`.

In [ ]:
import copy
from tqdm.auto import tqdm


def _get_protein_embs(data):
    """Extract per-graph protein embeddings from a batched PyG Data."""
    if not hasattr(data, 'protein_emb') or data.protein_emb is None:
        return None
    # For batched data, protein_emb is the embedding for each graph
    # We need to return a list of embeddings
    try:
        # protein_emb stored per graph — extract per batch element
        batch_ids = data.batch.unique()
        embs = []
        for b_id in batch_ids:
            # Each graph stores its own protein_emb
            # For InMemoryDataset, this is trickier with batching
            # Fallback: use the global protein_emb
            embs.append(data.protein_emb.to(data.x.device))
        return embs
    except Exception:
        return None


best_val_loss = float('inf')
best_model_weights = None
patience_counter = 0

EPOCHS = 100
PATIENCE = 15

for epoch in range(1, EPOCHS + 1):
    # --- TRAIN ---
    model.train()
    train_loss = 0
    num_train_graphs = 0
    for data in tqdm(train_loader, desc=f'Epoch {epoch} Train'):
        data = data.to(device)
        optimizer.zero_grad()
        protein_embs = _get_protein_embs(data)
        out = model(
            data.x.float(),
            data.pos.float(),
            data.batch,
            protein_embs=protein_embs,
        )
        loss = criterion(out.squeeze(), data.y.float())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * data.num_graphs
        num_train_graphs += data.num_graphs
    
    train_loss /= num_train_graphs
    
    # --- VALIDATION ---
    model.eval()
    val_loss = 0
    num_val_graphs = 0
    with torch.no_grad():
        for data in val_loader:
            data = data.to(device)
            protein_embs = _get_protein_embs(data)
            out = model(
                data.x.float(),
                data.pos.float(),
                data.batch,
                protein_embs=protein_embs,
            )
            loss = criterion(out.squeeze(), data.y.float())
            val_loss += loss.item() * data.num_graphs
            num_val_graphs += data.num_graphs
            
    val_loss /= num_val_graphs
    scheduler.step(val_loss)
    
    print(f'Epoch {epoch:03d} | Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f}')
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_weights = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

## 6. Evaluate on Test Set

In [ ]:
import numpy as np
import scipy.stats as stats

if best_model_weights is not None:
    model.load_state_dict(best_model_weights)
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        protein_embs = _get_protein_embs(data)
        out = model(
            data.x.float(),
            data.pos.float(),
            data.batch,
            protein_embs=protein_embs,
        )
        all_preds.extend(out.cpu().squeeze().tolist())
        all_targets.extend(data.y.cpu().squeeze().tolist())

preds = np.array(all_preds)
targets = np.array(all_targets)
rmse = np.sqrt(np.mean((preds - targets) ** 2))
mae = np.mean(np.abs(preds - targets))
pearson_r, _ = stats.pearsonr(preds, targets)
spearman_r, _ = stats.spearmanr(preds, targets)

print('\n' + '='*40)
print('FINAL TEST SET METRICS (Protein-Aware EGNN)')
print('='*40)
print(f'RMSE:          {rmse:.4f} pKa units')
print(f'MAE:           {mae:.4f} pKa units')
print(f'Pearson r:     {pearson_r:.4f}')
print(f'Spearman rho:  {spearman_r:.4f}')
print('='*40)

## 7. Export Trained Weights

In [ ]:
import json as _json

# Save with protein_encoder weights included
torch.save(best_model_weights, 'gnn_predictor_protein_aware.pth')

train_log = {
    'model': 'GNNPredictor (E(n) Equivariant + ESM-2 cross-attention)',
    'esm2_model': 'facebook/esm2_t33_650M_UR50D',
    'protein_aware': True,
    'best_val_mse': best_val_loss,
    'test_rmse': float(rmse),
    'test_pearson_r': float(pearson_r),
    'test_spearman_r': float(spearman_r),
    'atom_feat_dim': 43,
    'hidden_dim': 128,
    'n_layers': 4,
}
with open('training_log_egnn_protein.json', 'w') as f:
    _json.dump(train_log, f, indent=2)

print(f'Saved protein-aware weights to gnn_predictor_protein_aware.pth')
print(f'To use: copy to models/gnn_predictor.pth in your VLM project')
print(f'The load_model() function auto-detects protein encoder weights.')